# 📊 E-Commerce Customer Analytics & RFM Segmentation (Olist Dataset)

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta

df_customers = pd.read_csv('../data/raw/customers.csv')
df_orders = pd.read_csv('../data/raw/orders.csv')
df_items = pd.read_csv('../data/raw/order_items.csv')
df_reviews = pd.read_csv('../data/raw/order_reviews.csv')

print(f"Customers: {len(df_customers):,} | Orders: {len(df_orders):,} | Items: {len(df_items):,}")

In [ ]:
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_delivered = df_orders[df_orders['order_status'] == 'delivered']

df_merged = df_delivered.merge(df_items, on='order_id').merge(df_customers, on='customer_id')
print(f"Total Revenue: {df_merged['price'].sum():,.2f} BRL")
print(f"AOV: {df_merged['price'].sum() / df_delivered['order_id'].nunique():,.2f} BRL")

In [ ]:
snapshot_date = df_merged['order_purchase_timestamp'].max() + timedelta(days=1)
rfm = df_merged.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'price': 'sum'
}).reset_index()
rfm.columns = ['customer_unique_id', 'Recency', 'Frequency', 'Monetary']

rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = rfm['Frequency'].apply(lambda x: 5 if x>=4 else (4 if x==3 else (3 if x==2 else 1)))
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 4 and f >= 4 and m >= 4: return 'Champions'
    elif r >= 3 and f >= 3: return 'Loyal Customers'
    elif r <= 2 and f >= 3: return 'At Risk'
    elif r <= 2 and f <= 2 and m <= 2: return 'Hibernating'
    else: return 'Others'

rfm['Segment'] = rfm.apply(segment, axis=1)
rfm['Segment'].value_counts()